<a id="top"></a>

# Bell-pair workbench: the tutorial

This tutorial takes a Bell pair from the idea to a circuit on a processor model, ready to run. Part 1 follows the route a first course takes, from a state vector to an error check, and writes each step in Qiskit. Part 2 is the system you work in: a circuit before it runs, what the compiler makes of it, and the views you open when something looks wrong.

**By the end you'll be able to:**

- see where a circuit makes the pair and how it carries it to the ports, on the processor graph;
- check that a circuit leaves an exact pair on the ports before spending a single noisy shot;
- read what compiling costs: gates by type, two-qubit gates per coupler, depth and duration;
- take a circuit of your own from Qiskit to a comparison with the committed runs.

**You should already know** superposition, entanglement and interference; linear algebra at the level of vectors, matrices, the tensor product and eigenvalues; and enough Python to run a cell. The benchmark's own terms, such as port, correlator and acceptance, are defined in [CHALLENGE.md's Terms](../docs/CHALLENGE.md#terms).

**How to use it.** Run the next cell first, then start where you are. New to Qiskit circuits: [Part 1](#part-1). New to the benchmark: [Part 2](#part-2). Ready to work: open the [workbench notebook](workbench.ipynb). In JupyterLab and Colab, each part folds by its heading.

| Section | What you'll see |
|---|---|
| [Part 1: from a state vector to a checked Bell pair](#part-1) | Eight lessons: the algebra with one worked example, then the same thing in Qiskit |
| [Part 2: working with the benchmark](#part-2) | How a circuit meets the benchmark and our system, step by step, on one example |
| [Part 3: the workbench](workbench.ipynb), a notebook of its own | The cells you edit and run for your own circuit, and the steps that keep it |

**Run the next cell first, whichever part you start in.** It loads Qiskit, the package and the lesson views.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display
from qiskit import ClassicalRegister, QuantumCircuit
from qiskit.quantum_info import Operator, Pauli, Statevector

from bellbench.circuits import load_circuit
from bellbench.fidelity import bell_fidelity, fidelity_from_counts
from bellbench.views.lessons import algebra, bloch_steps, journey, ket, matrix_of, state_steps, tex, whole
from bellbench.workbench import Workbench, builders

<a id="part-1"></a>

# Part 1: from a state vector to a checked Bell pair

**Part 1 follows the route a first course in quantum information takes, and writes each step in Qiskit.** Every lesson has the same four parts: its point, the algebra with one worked example, the same thing in Qiskit, and where the benchmark uses it. No benchmark code yet: Part 2 brings that in.

| Lesson | The idea | The benchmark uses it for |
|---|---|---|
| [1.1](#lesson-1-1) | A state is a vector | Every qubit starting in \|0⟩ |
| [1.2](#lesson-1-2) | A gate is a unitary matrix | The gates a circuit is written and compiled in |
| [1.3](#lesson-1-3) | Measurement and shots | Every result being counts over shots |
| [1.4](#lesson-1-4) | Two qubits | The two ports inside a larger register |
| [1.5](#lesson-1-5) | The Bell pair | The target state |
| [1.6](#lesson-1-6) | Is it the pair? Correlations and F | The fidelity every run reports |
| [1.7](#lesson-1-7) | Moving it | The transports |
| [1.8](#lesson-1-8) | Errors and catching them | The protected cases and the acceptance |

<a id="lesson-1-1"></a>

## 1.1 A state is a vector

**A qubit's state is a unit vector with two complex entries, and the squared size of each entry is the chance of reading 0 or 1.**

**The algebra.** The two states a reading can give are the basis vectors

$$|0\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}, \qquad |1\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix},$$

and a general state is a superposition of them, with complex amplitudes $\alpha$ and $\beta$:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle = \begin{pmatrix} \alpha \\ \beta \end{pmatrix}, \qquad \langle\psi|\psi\rangle = |\alpha|^2 + |\beta|^2 = 1$$

The bra $\langle\psi|$ is the conjugate transpose of the ket, so $\langle\phi|\psi\rangle$ is the inner product of two states. Reading the qubit gives 0 with probability $|\langle 0|\psi\rangle|^2 = |\alpha|^2$ and 1 with probability $|\langle 1|\psi\rangle|^2 = |\beta|^2$.

**Example.** Two states with the same probabilities:

$$|{+}\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ 1 \end{pmatrix}, \qquad |{+i}\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ i \end{pmatrix}, \qquad |\alpha|^2 = |\beta|^2 = \frac{1}{2} \text{ for both}$$

They differ only in the phase between their two entries. A reading in 0 and 1 cannot see that phase; [1.3](#lesson-1-3) shows the reading that can.

**In Qiskit,** a `Statevector` holds the vector. The cell builds the three states and takes their norm and probabilities from it.

In [ ]:
states = {
    "|0⟩": Statevector.from_label("0"),
    "|+⟩": Statevector(np.array([1, 1]) / np.sqrt(2)),
    "|+i⟩": Statevector(np.array([1, 1j]) / np.sqrt(2)),
}
for name, state in states.items():
    norm = np.vdot(state.data, state.data).real      # ⟨ψ|ψ⟩
    read_0, read_1 = state.probabilities()           # |α|² and |β|²
    algebra(rf"\text{{{name}}} = {tex(state.data)} \qquad \langle\psi|\psi\rangle = {norm:.2f} \qquad "
            rf"P(0) = {read_0:.2f} \qquad P(1) = {read_1:.2f}")

**Where the benchmark uses it.** Every qubit of a processor model starts in $|0\rangle$, and a circuit is a sequence of changes to this vector. Several steps rely on idle qubits still being in $|0\rangle$, which [1.7](#lesson-1-7) makes exact.

<a id="lesson-1-2"></a>

## 1.2 A gate is a unitary matrix

**A gate multiplies the state vector by a unitary matrix, which keeps its length 1; on one qubit, that is a rotation of an arrow on the Bloch sphere.**

**The algebra.** A matrix $U$ is unitary when $U^\dagger U = I$. Then $\langle\psi|U^\dagger U|\psi\rangle = \langle\psi|\psi\rangle = 1$, so a gate turns one valid state into another and can always be undone by $U^\dagger$. Four gates do most of the work here:

$$X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}, \qquad Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}, \qquad H = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}, \qquad S = \begin{pmatrix} 1 & 0 \\ 0 & i \end{pmatrix}$$

X flips $|0\rangle$ and $|1\rangle$, a bit flip. Z changes the sign of $|1\rangle$, a phase flip. H makes an equal superposition, and S turns the phase by a quarter turn.

**Example.** H then S on $|0\rangle$, one matrix product each:

$$H|0\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}\begin{pmatrix} 1 \\ 0 \end{pmatrix} = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ 1 \end{pmatrix} = |{+}\rangle, \qquad S|{+}\rangle = \begin{pmatrix} 1 & 0 \\ 0 & i \end{pmatrix}\frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ 1 \end{pmatrix} = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ i \end{pmatrix} = |{+i}\rangle$$

**The Bloch sphere.** A global phase changes no reading, so every one-qubit state can be written with two angles:

$$|\psi\rangle = \cos\frac{\theta}{2}\,|0\rangle + e^{i\phi}\sin\frac{\theta}{2}\,|1\rangle$$

$\theta$ is the angle down from the vertical axis and $\phi$ the angle around it, so the state is a point on a sphere: $|0\rangle$ at the top, $|1\rangle$ at the bottom, $|{+}\rangle$ at $\theta = \pi/2, \phi = 0$ on the x axis, and $|{+i}\rangle$ at $\theta = \pi/2, \phi = \pi/2$ on the y axis. H carried the arrow from the top to +x, and S turned it around the vertical axis to +y.

**In Qiskit,** a `QuantumCircuit` lists the gates and an `Operator` gives a gate's matrix. The first cell takes the four matrices from Qiskit and checks each is unitary. The second writes the example as a circuit and steps through it on the sphere.

In [ ]:
for name in ("x", "z", "h", "s"):
    gate = QuantumCircuit(1)
    getattr(gate, name)(0)                           # the one gate, on qubit 0
    matrix = Operator(gate).data
    unitary = np.allclose(matrix.conj().T @ matrix, np.eye(2))
    algebra(rf"{name.upper()} = {tex(matrix)} \qquad U^\dagger U = I\text{{: {'yes' if unitary else 'NO'}}}")

In [ ]:
one = QuantumCircuit(1)    # one qubit, starting in |0⟩
one.h(0)                   # from the top of the sphere to +x
one.s(0)                   # a quarter turn around the vertical axis, to +y
display(one.draw("mpl"))
bloch_steps(one, notes=[
    "The qubit starts in |0⟩: θ = 0, the top of the sphere.",
    "H gives (|0⟩ + |1⟩)/√2: θ = π/2 and φ = 0, the +x axis.",
    "S multiplies the |1⟩ entry by i: θ stays π/2 and φ becomes π/2, the +y axis. The chances of reading 0 and 1 "
    "have not changed; only the phase has.",
])

**Where the benchmark uses it.** A circuit is written in gates like H and CX. Before it runs, the compiler rewrites it in the processor's own gates, which are also unitary matrices; [2.4](#section-2-4) counts both sets.

<a id="lesson-1-3"></a>

## 1.3 Measurement and shots

**A reading gives one eigenvalue of the measured operator, at random; many shots estimate its average, with an uncertainty that shrinks as $1/\sqrt{N}$.**

**The algebra.** A reading in 0 and 1 measures the operator Z. Its eigenvectors are $|0\rangle$ with eigenvalue +1 and $|1\rangle$ with eigenvalue −1, so "reads 0" means +1 and "reads 1" means −1. By the measurement postulate, a state $|\psi\rangle$ gives the eigenvalue of eigenvector $|v\rangle$ with probability $|\langle v|\psi\rangle|^2$, and is left in $|v\rangle$ afterwards. The average result is the expectation value:

$$\langle Z\rangle = \langle\psi|Z|\psi\rangle = P(0) - P(1)$$

One shot gives one eigenvalue, not the average. From $N$ shots with $N_0$ zeros and $N_1$ ones, the estimate and its standard error are

$$\langle Z\rangle \approx \frac{N_0 - N_1}{N}, \qquad \text{standard error} = \sqrt{\frac{1 - \langle Z\rangle^2}{N}}$$

**Reading along x or y.** A processor reads only Z, so another operator is measured by rotating first. X has eigenvectors $|{+}\rangle$ and $|{-}\rangle$, and H maps them onto $|0\rangle$ and $|1\rangle$, which is the identity $HZH = X$: apply H, then read Z, and you have read X. For Y, apply $S^\dagger$ then H, since $S\,HZH\,S^\dagger = Y$.

**Example.** For $|{+}\rangle$, $\langle Z\rangle = 0$ but $\langle X\rangle = +1$: random in 0 and 1, certain along x. For $|{+i}\rangle$, both $\langle Z\rangle$ and $\langle X\rangle$ are 0 and $\langle Y\rangle = +1$. The phase that 0 and 1 could not see in [1.1](#lesson-1-1) decides the reading along x and y.

**In Qiskit,** the cell reads $|{+}\rangle$ along z and along x, with 100 shots and then 10,000, and sets each estimate beside the exact value.

In [ ]:
plus = Statevector.from_label("+")
turn_x_to_z = QuantumCircuit(1)
turn_x_to_z.h(0)                                     # H before the reading: reads X instead of Z

rows = []
for reading, state, exact in (("Z", plus, 0), ("X, by H then Z", plus.evolve(turn_x_to_z), 1)):
    for shots in (100, 10_000):
        state.seed(2026)                             # a fixed seed, so the table repeats
        counts = state.sample_counts(shots)          # {"0": N0, "1": N1}
        zeros = counts.get("0", 0)
        estimate = (2 * zeros - shots) / shots       # (N0 − N1) / N
        error = np.sqrt((1 - estimate**2) / shots)
        rows.append({"reading": reading, "shots": f"{shots:,}", "zeros": zeros, "ones": shots - zeros,
                     "estimate ± standard error": f"{estimate:+.3f} ± {error:.3f}".replace("-", "−"),
                     "exact": f"{exact:+d}".replace("+0", "0")})
display(HTML(pd.DataFrame(rows).to_html(index=False, border=0)))

**Reading it.** Along z the estimate lands near 0 and its standard error falls from about 0.1 to about 0.01 as the shots go from 100 to 10,000. Along x every shot reads 0, so the estimate is exactly +1 with no spread.

**Where the benchmark uses it.** Every number a run reports is counts over shots, 20,000 per reading in the committed benchmarks, and every F comes with a standard error built from this formula ([2.5](#section-2-5)).

<a id="lesson-1-4"></a>

## 1.4 Two qubits

**Two qubits share one state vector with four entries, and a two-qubit gate is a 4 by 4 unitary; CNOT flips the second qubit wherever the first is 1.**

**The algebra.** The space of two qubits is the tensor product of their spaces. Its basis states, written with qubit 0 first, are

$$|00\rangle = \begin{pmatrix} 1 \\ 0 \\ 0 \\ 0 \end{pmatrix}, \quad |01\rangle = \begin{pmatrix} 0 \\ 1 \\ 0 \\ 0 \end{pmatrix}, \quad |10\rangle = \begin{pmatrix} 0 \\ 0 \\ 1 \\ 0 \end{pmatrix}, \quad |11\rangle = \begin{pmatrix} 0 \\ 0 \\ 0 \\ 1 \end{pmatrix}, \qquad |a\rangle\otimes|b\rangle = \begin{pmatrix} a_0 b_0 \\ a_0 b_1 \\ a_1 b_0 \\ a_1 b_1 \end{pmatrix}$$

A gate on one qubit acts as its matrix tensored with the identity on the other, such as $H\otimes I$ for H on qubit 0. CNOT, written CX in Qiskit, has a control and a target: it leaves the target alone when the control is 0 and applies X to it when the control is 1. With qubit 0 as control it exchanges $|10\rangle$ and $|11\rangle$:

$$\mathrm{CX} = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \end{pmatrix}, \qquad |00\rangle \to |00\rangle, \quad |01\rangle \to |01\rangle, \quad |10\rangle \to |11\rangle, \quad |11\rangle \to |10\rangle$$

It is a permutation of the basis states, so it is reversible, and applying it twice gives the identity.

**Qiskit orders the basis the other way,** with qubit 0 as the last digit of a bit string. Its vectors and matrices therefore list our $|10\rangle$ in second place and our $|01\rangle$ in third. This tutorial reorders to qubit 0 first everywhere, and says so wherever Qiskit's own order shows.

**In Qiskit,** the cell takes $H\otimes I$ and CX from circuits, shows CX in both orders, and builds CX's truth table by applying it to each basis state.

In [ ]:
h_on_0 = QuantumCircuit(2)
h_on_0.h(0)
cx_0_to_1 = QuantumCircuit(2)
cx_0_to_1.cx(0, 1)                                   # control qubit 0, target qubit 1
algebra(
    rf"H\otimes I = {tex(matrix_of(h_on_0))} \qquad \mathrm{{CX}} = {tex(matrix_of(cx_0_to_1))}",
    rf"\text{{The same CX as Qiskit prints it, qubit 0 last:}}\quad {tex(Operator(cx_0_to_1).data)}",
)
rows = []
for label in ("00", "01", "10", "11"):               # written qubit 0 first
    start = Statevector.from_label(label[::-1])      # Qiskit's labels put qubit 0 last
    rows.append({"in": f"|{label}⟩", "out": ket(start.evolve(cx_0_to_1), [0, 1])})
display(HTML(pd.DataFrame(rows).to_html(index=False, border=0)))

**Where the benchmark uses it.** The two ports are two qubits inside a register of 5, 9 or 13. Every ket in these notebooks reads qubit 0 first, left to right, while the raw counts a run stores keep Qiskit's order.

<a id="lesson-1-5"></a>

## 1.5 The Bell pair

**H then CNOT turn $|00\rangle$ into $(|00\rangle + |11\rangle)/\sqrt{2}$, a state that cannot be written as one qubit's state times the other's: the two qubits are entangled.**

**The algebra.** H on qubit 0, then CX with qubit 0 as control:

$$(H\otimes I)\,|00\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ 0 \\ 1 \\ 0 \end{pmatrix} = \frac{|0\rangle + |1\rangle}{\sqrt{2}}\otimes|0\rangle, \qquad \mathrm{CX}\,\frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ 0 \\ 1 \\ 0 \end{pmatrix} = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ 0 \\ 0 \\ 1 \end{pmatrix} = \frac{|00\rangle + |11\rangle}{\sqrt{2}} = |\Phi^+\rangle$$

After H the state still factors into a state of qubit 0 times a state of qubit 1, so the two are independent. After CX it does not. A product state is

$$(a|0\rangle + b|1\rangle)\otimes(c|0\rangle + d|1\rangle) = ac\,|00\rangle + ad\,|01\rangle + bc\,|10\rangle + bd\,|11\rangle,$$

and matching $|\Phi^+\rangle$ needs $ad = 0$ and $bc = 0$ while $ac \neq 0$ and $bd \neq 0$. No four numbers do that. A state that cannot be factored is entangled.

**Reading it.** The probabilities are the squared amplitudes: 00 half the time, 11 half the time, and never 01 or 10. Reading qubit 0 as 0 leaves $|00\rangle$, so qubit 1 then reads 0 for certain; reading 1 leaves $|11\rangle$. Each qubit alone is random, and the two always agree.

**The four Bell states.** $|\Phi^+\rangle$ is one of four entangled states that form a basis of the two-qubit space. The other three are what one error on qubit 0 makes of it:

$$|B_{00}\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}} = |\Phi^+\rangle, \qquad |B_{01}\rangle = \frac{|01\rangle + |10\rangle}{\sqrt{2}}, \qquad |B_{10}\rangle = \frac{|00\rangle - |11\rangle}{\sqrt{2}}, \qquad |B_{11}\rangle = \frac{|01\rangle - |10\rangle}{\sqrt{2}}$$

A bit flip, X, gives $|B_{01}\rangle$; a phase flip, Z, gives $|B_{10}\rangle$; both give $|B_{11}\rangle$, up to an overall sign, a global phase that no reading can see. [1.6](#lesson-1-6) and [1.8](#lesson-1-8) build on this.

**In Qiskit,** the first cell writes the two gates and steps through them: the ket and each outcome's probability after every gate. The second applies each error to the pair and prints the state it leaves.

In [ ]:
pair = QuantumCircuit(2)   # two qubits, both starting in |0⟩
pair.h(0)                  # qubit 0 into an equal superposition
pair.cx(0, 1)              # flip qubit 1 wherever qubit 0 is 1
display(pair.draw("mpl"))
state_steps(pair, notes=[
    "Both qubits start in |0⟩: the only outcome is 00.",
    "H on qubit 0: outcomes 00 and 10, half each. Qubit 1 is still 0 whatever qubit 0 reads; the state factors.",
    "CX: outcomes 00 and 11, half each, and never 01 or 10. The state no longer factors: the two qubits are entangled.",
])

In [ ]:
errors = {"no error": [], "bit flip, X on qubit 0": ["x"], "phase flip, Z on qubit 0": ["z"],
          "both, X then Z on qubit 0": ["x", "z"]}
bell_states = {}
for name, gates in errors.items():
    circuit = pair.copy()
    for gate in gates:
        getattr(circuit, gate)(0)                    # the error, applied by hand
    bell_states[name] = Statevector(circuit)
    display(Markdown(f"{name}: {ket(bell_states[name], [0, 1])}"))

**Where the benchmark uses it.** $|\Phi^+\rangle$ on the two ports is the target of every circuit in the benchmark. A correct circuit, run without noise, ends in exactly this state on its ports.

<a id="lesson-1-6"></a>

## 1.6 Is it the pair? Correlations and F

**The fidelity F is the probability of finding the two ports in $|\Phi^+\rangle$, and a run gets it from three correlations, each measured by its own circuit: ZZ, XX and YY.**

**The algebra.** By the measurement postulate of [1.3](#lesson-1-3), a state $|\psi\rangle$ is found in $|\Phi^+\rangle$ with probability

$$F = |\langle\Phi^+|\psi\rangle|^2$$

The four Bell states are orthonormal, so F is 1 for $|B_{00}\rangle$ and 0 for the other three. For $|00\rangle$, which has no entanglement, $\langle\Phi^+|00\rangle = 1/\sqrt{2}$ and $F = 1/2$.

Measuring in the Bell basis directly would need a CX between the two ports, and the ports are not neighbours. Three readings that need no such gate give the same number. Reading both ports in Z measures $Z\otimes Z$, whose eigenvalue is +1 when the two results agree and −1 when they differ, so its average is a correlation:

$$\langle ZZ\rangle = \langle\psi|\,Z\otimes Z\,|\psi\rangle = P(\text{agree}) - P(\text{differ}),$$

and the same holds for $X\otimes X$ and $Y\otimes Y$, read by rotating each port first as in [1.3](#lesson-1-3). The link to F is one identity, the target's projector written in these operators:

$$|\Phi^+\rangle\langle\Phi^+| = \frac{1}{4}\left(I\otimes I + X\otimes X - Y\otimes Y + Z\otimes Z\right) \quad\Longrightarrow\quad F = \frac{1 + \langle XX\rangle - \langle YY\rangle + \langle ZZ\rangle}{4}$$

**Example.** $|\Phi^+\rangle$ agrees along z, agrees along x and differs along y: $\langle ZZ\rangle = +1$, $\langle XX\rangle = +1$, $\langle YY\rangle = -1$, so $F = (1 + 1 + 1 + 1)/4 = 1$. A phase flip turns it into $|B_{10}\rangle$, which flips the x and y correlations: $F = (1 - 1 - 1 + 1)/4 = 0$.

**A noisy run is a mix of shots.** If 80% of shots hold $|B_{00}\rangle$ and 20% hold $|B_{10}\rangle$, each correlation is the average over shots, and so is F: $0.8\times 1 + 0.2\times 0 = 0.8$.

**In Qiskit,** the first cell checks the identity and takes the three correlations and F for each state of [1.5](#lesson-1-5). The second draws the three circuits a run makes from one written circuit, then runs the 80 and 20 mix through them with shots and scores the counts with the benchmark's own function.

In [ ]:
target = bell_states["no error"]
projector = np.outer(target.data, target.data.conj())                # |Φ+⟩⟨Φ+|
pauli = {name: Pauli(name).to_matrix() for name in ("II", "XX", "YY", "ZZ")}
same = np.allclose(projector, (pauli["II"] + pauli["XX"] - pauli["YY"] + pauli["ZZ"]) / 4)
display(Markdown(f"|Φ+⟩⟨Φ+| equals (II + XX − YY + ZZ)/4: {'yes' if same else 'NO'}"))

rows = []
for name, state in (bell_states | {"|00⟩, no entanglement": Statevector.from_label("00")}).items():
    xx, yy, zz = (state.expectation_value(Pauli(basis)).real for basis in ("XX", "YY", "ZZ"))
    overlap = abs(np.vdot(target.data, state.data)) ** 2             # |⟨Φ+|ψ⟩|²
    rows.append({"state": name, "ket": ket(state, [0, 1]), "⟨ZZ⟩": whole(zz), "⟨XX⟩": whole(xx), "⟨YY⟩": whole(yy),
                 "F from the three": f"{bell_fidelity(xx, yy, zz):.2f}", "|⟨Φ+|ψ⟩|²": f"{overlap:.2f}"})
display(HTML(pd.DataFrame(rows).to_html(index=False, border=0)))

In [ ]:
def basis_circuit(prepare: QuantumCircuit, basis: str) -> QuantumCircuit:
    """`prepare`, then both qubits rotated so a Z reading measures `basis`, then read."""
    circuit = prepare.copy()
    for qubit in (0, 1):
        if basis == "YY":
            circuit.sdg(qubit)                       # S†, then H: reads Y
        if basis in ("XX", "YY"):
            circuit.h(qubit)                         # H: reads X
    circuit.measure_all()
    return circuit

for basis in ("ZZ", "XX", "YY"):
    display(Markdown(f"**{basis}**"), basis_circuit(pair, basis).draw("mpl"))

flipped = pair.copy()
flipped.z(0)                                         # the pair with a phase flip: |B10⟩
counts = {}
for basis in ("ZZ", "XX", "YY"):
    counts[basis] = {"00": 0, "01": 0, "10": 0, "11": 0}
    for prepare, shots, seed in ((pair, 8_000, 1), (flipped, 2_000, 2)):   # 80% of shots the pair, 20% flipped
        state = Statevector(basis_circuit(prepare, basis).remove_final_measurements(inplace=False))
        state.seed(seed)
        for outcome, count in state.sample_counts(shots).items():
            counts[basis][outcome] += count
score = fidelity_from_counts(counts)                 # the benchmark's own scoring of port counts
rows = [{"reading": basis, "agree": counts[basis]["00"] + counts[basis]["11"],
         "differ": counts[basis]["01"] + counts[basis]["10"],
         "correlation": f"{score.correlators[basis].value:+.3f}".replace("-", "−")} for basis in ("ZZ", "XX", "YY")]
display(HTML(pd.DataFrame(rows).to_html(index=False, border=0)))
display(Markdown(f"**F = {score.value:.4f} ± {score.standard_error:.4f}** from 10,000 shots per reading"))

**Reading it.** In the first table, "F from the three" and $|\langle\Phi^+|\psi\rangle|^2$ agree on every row: 1 for the target, 0 for the three Bell states an error makes, and 0.5 for $|00\rangle$. No unentangled state scores above 0.5 ([CHALLENGE.md, Verifying the pair](../docs/CHALLENGE.md#verifying-the-pair)). In the mix, the z reading always agrees, since a phase flip does not change it, while the x and y correlations fall from 1 to 0.6 in size, since a fifth of the shots now count against them, and F comes out at 0.8.

**Where the benchmark uses it.** Every run compiles these three circuits from the one you write, names them ZZ, XX and YY, and reports each correlation and F with its standard error ([2.5](#section-2-5)). A run folder's `circuit-zz.png` is the first of the three, compiled.

<a id="lesson-1-7"></a>

## 1.7 Moving it

**The ports are not neighbours, so the pair is made beside one port and its far half is carried along a path of couplers; a SWAP costs three CX gates, and a move onto a qubit in $|0\rangle$ costs two.**

**The algebra.** Call two neighbouring qubits a and b, and order the basis $|ab\rangle$. A SWAP exchanges $|01\rangle$ and $|10\rangle$, and it equals three CX gates:

$$\mathrm{SWAP} = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \end{pmatrix} = \mathrm{CX}_{a\to b}\;\mathrm{CX}_{b\to a}\;\mathrm{CX}_{a\to b}$$

A move is the first two of the three. Take a in any state and b in $|0\rangle$:

$$(\alpha|0\rangle + \beta|1\rangle)\otimes|0\rangle = \alpha|00\rangle + \beta|10\rangle \;\xrightarrow{\;\mathrm{CX}_{a\to b}\;}\; \alpha|00\rangle + \beta|11\rangle \;\xrightarrow{\;\mathrm{CX}_{b\to a}\;}\; \alpha|00\rangle + \beta|01\rangle = |0\rangle\otimes(\alpha|0\rangle + \beta|1\rangle)$$

The state has moved to b, and a is back in $|0\rangle$. The third CX would have a as its control, and a is $|0\rangle$, so it would do nothing. As matrices, the move and the SWAP agree on the columns for $|00\rangle$ and $|10\rangle$, the inputs with b in $|0\rangle$, and differ on the other two. A move onto a qubit that is not in $|0\rangle$ is a different operation.

**Example.** On ibm2016, a T of five qubits, ports 0 and 4 are two hops apart, through qubit 3. Make the pair on 0 and 3 as in [1.5](#lesson-1-5), then carry the half on 3 to port 4. With a SWAP the circuit has 4 two-qubit gates; with a move it has 3.

**In Qiskit,** the first cell checks the three matrix statements. The second writes both transports and steps through each on the processor graph: white is a qubit on its own, dark a qubit that is part of an entangled state, and orange marks the gate just applied.

In [ ]:
swap = QuantumCircuit(2)                             # qubit 0 is a, qubit 1 is b
swap.swap(0, 1)
three_cx = QuantumCircuit(2)
three_cx.cx(0, 1)
three_cx.cx(1, 0)
three_cx.cx(0, 1)
move = QuantumCircuit(2)
move.cx(0, 1)                                        # copy a onto b...
move.cx(1, 0)                                        # ...then clear a
algebra(rf"\mathrm{{SWAP}} = {tex(matrix_of(swap))} \qquad "
        rf"\text{{move}} = \mathrm{{CX}}_{{b\to a}}\,\mathrm{{CX}}_{{a\to b}} = {tex(matrix_of(move))}")
b_in_zero = [0, 2]                                   # the columns for |00⟩ and |10⟩, the inputs with b in |0⟩
checks = {
    "three CX equal a SWAP": np.allclose(matrix_of(three_cx), matrix_of(swap)),
    "a move equals a SWAP when b starts in |0⟩": np.allclose(matrix_of(move)[:, b_in_zero], matrix_of(swap)[:, b_in_zero]),
    "a move equals a SWAP on every input": np.allclose(matrix_of(move), matrix_of(swap)),
}
display(Markdown(" · ".join(f"{claim}: {'yes' if holds else 'no'}" for claim, holds in checks.items())))

In [ ]:
swap_forward = QuantumCircuit(5)  # all five qubits of ibm2016, each starting in |0⟩
swap_forward.h(0)                 # the pair on port 0 and its neighbour 3
swap_forward.cx(0, 3)
swap_forward.swap(3, 4)           # carry the half on 3 to port 4 with a SWAP

move_chain = QuantumCircuit(5)
move_chain.h(0)                   # the same pair
move_chain.cx(0, 3)
move_chain.cx(3, 4)               # copy the half on 3 onto 4, which is in |0⟩...
move_chain.cx(4, 3)               # ...then clear it from 3: a move

transports = {"swap_forward": swap_forward, "move_chain": move_chain}
rows = []
for name, circuit in transports.items():
    display(Markdown(f"**{name}**"), circuit.draw("mpl", idle_wires=False))
    journey(circuit, "ibm2016", f"{name} on ibm2016")
    rows.append({"transport": name, "two-qubit gates, a SWAP as 3 CX":
                 circuit.count_ops().get("cx", 0) + 3 * circuit.count_ops().get("swap", 0)})
display(HTML(pd.DataFrame(rows).to_html(index=False, border=0)))

**Reading it.** Both transports end in $(|000\rangle + |101\rangle)/\sqrt{2}$ on qubits 0, 3 and 4: the pair on the ports, with qubit 3 back in $|0\rangle$ and white again. The SWAP gets there in one step. The move passes through $(|000\rangle + |111\rangle)/\sqrt{2}$, all three qubits entangled, because its first CX copies the half onto qubit 4 before its second clears it from qubit 3.

**Where the benchmark uses it.** These are two of its transports, and the cost a run reports is its two-qubit gates ([2.4](#section-2-4)). The move works only because qubit 4 started in $|0\rangle$, which is why every result here rests on idle qubits starting in $|0\rangle$.

<a id="lesson-1-8"></a>

## 1.8 Errors and catching them

**A check measures a stabilizer of the pair onto an extra qubit, so a shot with an error it can see is thrown away; it detects errors, never corrects them, and is blind to any error that leaves its stabilizer alone.**

**The errors.** In a noisy run, each gate, each wait and each reading has a small chance of going wrong; the noise profile sets those chances. On one qubit the basic errors are the two gates of [1.2](#lesson-1-2) applied when nobody asked: a bit flip, X, and a phase flip, Z. From [1.5](#lesson-1-5), on one half of the pair they turn $|B_{00}\rangle$ into $|B_{01}\rangle$ and $|B_{10}\rangle$.

**The algebra.** An operator that leaves a state unchanged is called a stabilizer of it. $|\Phi^+\rangle$ has two:

$$(X\otimes X)\,|\Phi^+\rangle = +|\Phi^+\rangle, \qquad (Z\otimes Z)\,|\Phi^+\rangle = +|\Phi^+\rangle$$

An error shows up when it flips a stabilizer's eigenvalue to −1.

- **A phase flip, Z on one qubit,** anticommutes with $X\otimes X$, since $ZX = -XZ$. So $(X\otimes X)(Z\otimes I)|\Phi^+\rangle = -(Z\otimes I)(X\otimes X)|\Phi^+\rangle = -(Z\otimes I)|\Phi^+\rangle$: the damaged state has eigenvalue −1, and an XX check flags it.
- **A bit flip, X on one qubit,** commutes with $X\otimes X$, so the eigenvalue stays +1 and an XX check misses it. It flips $Z\otimes Z$ instead, which only a ZZ check would catch.

**Reading a stabilizer onto an extra qubit.** The check must not read the pair itself, or it would collapse it. It uses an extra qubit: H puts it in $(|0\rangle + |1\rangle)/\sqrt{2}$, two CX gates apply $X\otimes X$ to the pair only in its $|1\rangle$ branch, and a second H brings the answer back. Writing the extra qubit first, for this one line:

$$\frac{|0\rangle + |1\rangle}{\sqrt{2}}\otimes|\psi\rangle \;\longrightarrow\; \frac{|0\rangle\otimes|\psi\rangle + |1\rangle\otimes(X\otimes X)|\psi\rangle}{\sqrt{2}} \;\xrightarrow{\;H\;}\; |0\rangle\otimes\frac{I + X\otimes X}{2}\,|\psi\rangle \;+\; |1\rangle\otimes\frac{I - X\otimes X}{2}\,|\psi\rangle$$

$(I + X\otimes X)/2$ and $(I - X\otimes X)/2$ project onto the +1 and −1 eigenspaces. So the extra qubit reads 0 when the pair has eigenvalue +1, and 1 when it has −1. This is the syndrome measurement of an error-correcting code.

**Detection, not correction.** A code that corrects, such as the three-qubit bit-flip code, follows its syndrome with a recovery gate. Here nothing is repaired: a shot whose check reads 1 is thrown away. The share of shots kept is the acceptance, and F of the kept shots is always reported with it.

**Example.** Suppose 20% of shots carry an error. If it is a phase flip, the check flags exactly those shots: before the check F is 0.8; of the kept shots F is 1, at an acceptance of 0.8. If it is a bit flip, the check flags nothing: F stays 0.8 with every shot kept.

**In Qiskit,** the cell builds the check, takes both stabilizers' eigenvalues and what the check reads for each error, then works the 20% example.

In [ ]:
check = QuantumCircuit(3)  # qubits 0 and 1 hold the pair; qubit 2 is the check's extra qubit
check.h(2)
check.cx(2, 0)             # X on qubit 0, only in the extra qubit's |1⟩ branch
check.cx(2, 1)             # and X on qubit 1
check.h(2)
display(check.draw("mpl"))

cases, rows = {}, []
for name, error in {"no error": None, "phase flip, Z on qubit 0": "z", "bit flip, X on qubit 0": "x"}.items():
    case = QuantumCircuit(3)
    case.h(0)
    case.cx(0, 1)                                    # the pair, with the extra qubit in |0⟩
    if error:
        getattr(case, error)(0)                      # the error, applied by hand
    state = Statevector(case)
    reads_1 = Statevector(case.compose(check)).probabilities([2])[1]   # how often the extra qubit reads 1
    fidelity = bell_fidelity(*(state.expectation_value(Pauli(basis), [0, 1]).real for basis in ("XX", "YY", "ZZ")))
    cases[name] = {"F": fidelity, "flagged": reads_1}
    rows.append({"case": name, "state on qubits 0, 1, 2": ket(state, [0, 1, 2]),
                 "⟨XX⟩": whole(state.expectation_value(Pauli("XX"), [0, 1]).real),
                 "⟨ZZ⟩": whole(state.expectation_value(Pauli("ZZ"), [0, 1]).real),
                 "check reads 1": f"{reads_1:.0%}", "F": f"{fidelity:.2f}"})
display(HTML(pd.DataFrame(rows).to_html(index=False, border=0)))

rows = []
for name in ("phase flip, Z on qubit 0", "bit flip, X on qubit 0"):
    clean, hit = cases["no error"], cases[name]
    kept = 0.8 * (1 - clean["flagged"]) + 0.2 * (1 - hit["flagged"])              # the acceptance
    f_kept = (0.8 * (1 - clean["flagged"]) * clean["F"] + 0.2 * (1 - hit["flagged"]) * hit["F"]) / kept
    rows.append({"error in 20% of shots": name, "F, every shot": f"{0.8 * clean['F'] + 0.2 * hit['F']:.2f}",
                 "acceptance": f"{kept:.2f}", "F, kept shots": f"{f_kept:.2f}"})
display(HTML(pd.DataFrame(rows).to_html(index=False, border=0)))

**Reading it.** With no error both stabilizers read +1 and the check reads 0. A phase flip turns $\langle XX\rangle$ to −1 and the check reads 1 every time. A bit flip leaves $\langle XX\rangle$ at +1, so the check never fires, though $\langle ZZ\rangle$ shows the damage. In the second table, discarding flagged shots lifts F from 0.80 to 1.00 for phase flips at the cost of a fifth of the shots, and does nothing for bit flips.

**Where the benchmark uses it.** The benchmark's XX check works this way through a bus of extra qubits ([2.1](#section-2-1)), and every protected result reports the shots each check flagged and the acceptance ([2.5](#section-2-5)). A check is described by what it cannot see as well as what it can.

<a id="part-2"></a>

# Part 2: working with the benchmark

**Part 2 shows how a circuit meets the benchmark and our system, step by step: a real benchmark circuit, the session's settings, making a circuit, checking it, running it, comparing it with the committed runs, the benchmarks themselves, and the path from trial to result.** One example circuit, written in Qiskit, runs through all of it; the [workbench notebook](workbench.ipynb) is where you do the same with your own. The tools print data only; the guide below says how to read each view.

<a id="reading-the-views"></a>
<details>
<summary><b>How to read each view</b></summary>

| View | What it shows | What to look for |
|---|---|---|
| Header | The circuit file and its label; the processor, its ports and the hops between them; how the shots are read; where the compiled circuits came from | A circuit that measures check bits is read by `parity` with its check groups; `postselect` reads every bit besides the ports as a check that must be 0 |
| Graph and written circuit | The couplers the written circuit uses, each with its gate count, beside the circuit; qubit numbers match the circuit's wires | The path the pair takes, and any coupler used more often than the transport needs |
| Cost line | Two-qubit gates in the compiled ZZ circuit, the benchmark's cost; SWAPs, and how many the router added; depth, counting gates only; the qubits used; the scheduled duration; whether routing left the pair on the ports | Router SWAPs above 0, or the pair not on the ports, mean compiling moved qubits |
| Gates by type | How often each gate appears in the written circuit and in the three compiled ones | Compiling turns each CX into one CZ and each SWAP into three, plus single-qubit rotations (rz, sx); XX and YY add the rotations that read those bases |
| `step=True` | The step-through from lessons [1.7](#lesson-1-7) and [2.1](#section-2-1), on the written circuit, ideal, with no noise, with the state as a ket beside each step | The last step's ket factors into the pair on the ports, (|00⟩ + |11⟩)/√2, times anything else; a check's qubits form a separate factor |
| `compiled=True` | The compiled ZZ circuit on physical qubits, as it runs, in short rows; a line each for XX and YY says how they differ from it. `compiled="all"` draws all three | The three share the same two-qubit gates; only the ports' single-qubit gates differ, because the compiler merges each basis's reading rotations into the rotations already there |
| `timeline=True` | When each qubit that acts is busy, in µs: a two-qubit gate, a single-qubit gate, a measurement, or waiting; then the same schedule as Qiskit draws it, gate by gate | Long waits on a qubit holding half of the pair: idle noise acts there |

| `run`, `result` | Warnings, if any; the headline: F of the kept shots with its standard error, the acceptance with its 95% Wilson interval, and F of every shot; a table per basis; what each check flagged; the cost | The ideal run must give F = 1 with every shot kept; F is never read without its acceptance; a check flagging far more in one basis than the others shows which error it catches |
| `compare` | A warning when the runs' noise settings differ; a table; a bar per run of its F, each labelled with its value and its two-qubit gates, with the first run's every-shot F as a grey band, a grey bar for every shot and a purple one for kept shots | A kept-shot bar above the band, at an acceptance you can afford; the axis does not start at 0, so compare the numbers on the bars; differing noise settings make a difference conditional on them |
| `benchmark` | One panel of bars per processor on a shared F axis, cases labelled, one colour per noise profile when there are several, then the case table; a benchmark without cases gets a run table | Compare within a panel: the same processor and noise, different circuits |

Each view prints a warning line only when something needs attention.

</details>

<a id="section-2-1"></a>

## 2.1 A real benchmark circuit

**On case 9's circuit the same view shows the pair crossing the hyperbolic graph and a check reaching both ports by a second path.** `move-chain-spare-xx-hyperbolic` makes the pair at port 0 and moves its far half along 0, 1, 9, 12. Its XX check then works on a spare path, 0, 3, 11, 12, which shares no qubit with the pair's, and reads the ports' XX parity there.

The cell loads the circuit file the benchmark ran, draws its circuit, then steps through it gate by gate, with the state as a ket and a note beside each step.

In [ ]:
bridge = load_circuit("move-chain-spare-xx-hyperbolic")  # the circuit file case 9 ran: gates, processor, analysis
display(bridge.circuit.draw("mpl", idle_wires=False, fold=40))
journey(bridge.circuit, bridge.processor, "move-chain-spare-xx-hyperbolic, case 9")

**Reading it.** In the stepper, the far half of the pair moves from qubit 1 to 9 to 12, and each middle qubit turns white again once the half has moved on. The last step's state says the rest. On qubits 0, 1, 3, 9, 11 and 12, it is (|000000⟩ + |001010⟩ + |100001⟩ + |101011⟩)/2, which factors into two pairs: (|00⟩ + |11⟩)/√2 on ports 0 and 12, and a separate (|00⟩ + |11⟩)/√2 on check qubits 3 and 11. The ports hold exactly the target, untouched by the check, which is why the check never discards a shot in a run without noise.

**Where it shows up:** Part 2 checks a circuit of your own the same way, in [2.4](#section-2-4).

## 2.2 Session settings

**One cell sets what this session compiles and runs with.** It matches the move-chain benchmark, so a circuit compiled here gets its committed run's transpiler seed.

In [ ]:
wb = Workbench(noise="boston-2026", shots=500, base_seed=20261116)  # the move-chain benchmark's profile and base seed

<a id="section-2-3"></a>

## 2.3 Making a circuit

**Write a circuit in Qiskit, name how its shots are read, and `build` turns it into a trial the workbench can check, run and compare.** The circuit below is written by hand on ibm2016. It carries the pair to ports 0 and 4 through qubit 3 with a move, as in [1.7](#lesson-1-7). Qubit 3 is then back in |0⟩ and sits next to both ports, so it doubles as the extra qubit of [1.8](#lesson-1-8)'s XX check. The check's bit goes in a register named for its role, `check_xx`, and the reading is `parity` on that one bit: keep a shot when it reads 0.

`build` checks the circuit as loading a circuit file would: one register of the processor's qubits, check registers named by their role, no measurement of the ports, no feedforward. It then saves the circuit as a trial in `workspace/circuits/`, which git ignores; building the same name again replaces it.

In [ ]:
CIRCUIT_NAME = "my-xx-check"  # the name the circuit keeps: promote and benchmark files use it
LABEL = "move to port 4, then an XX check through qubit 3"
PROCESSOR = "ibm2016"
ANALYSIS = {"name": "parity", "settings": {"checks": [["check_xx[0]"]]}}  # keep a shot when check_xx[0] reads 0

mine = QuantumCircuit(5)                  # ibm2016's five qubits; the run measures the ports itself
flag = ClassicalRegister(1, "check_xx")   # a check's register starts with its role, "check"
mine.add_register(flag)
mine.h(0)                                 # the pair on port 0 and qubit 3
mine.cx(0, 3)
mine.cx(3, 4)                             # move the half on 3 to port 4: qubit 3 is back in |0⟩
mine.cx(4, 3)
mine.h(3)                                 # 1.8's XX check, with qubit 3 as its extra qubit
mine.cx(3, 0)
mine.cx(3, 4)
mine.h(3)
mine.measure(3, flag[0])                  # 1 flags a phase flip on a port

trial = wb.build(CIRCUIT_NAME, mine, processor=PROCESSOR, analysis=ANALYSIS, label=LABEL)  # returns the trial's name

**A guard you will meet.** A circuit that measures a bit must say how that bit is read. `build` refuses it otherwise, because the default would treat the bit as a check that must read 0:

In [ ]:
try:
    wb.build(CIRCUIT_NAME, mine, processor=PROCESSOR, label=LABEL)  # no analysis named
except ValueError as refused:
    display(Markdown(f"Refused: {refused}"))

**Or let the builders write it.** The builders are a shortcut, not the only way: they write the same gates you would, with the bookkeeping done. They find a path between any processor's ports, track which qubits are in |0⟩ so a move is safe, refuse a step the circuit cannot take, and, when a check is added, record how its shots must be read. The cell checks them against [1.7](#lesson-1-7)'s move written by hand, then builds case 6's protected circuit from three calls, and hands the same steps to `build`, which saves the reading the builders wrote.

In [ ]:
hand = QuantumCircuit(5)   # 1.7's move, written by hand on ibm2016
hand.h(0)
hand.cx(0, 3)
hand.cx(3, 4)
hand.cx(4, 3)
built = builders.move_chain(builders.start("ibm2016"))  # the same transport, from the builders
display(Markdown(f"Same gates as the hand-written move: {'yes' if built.circuit == hand else 'NO'}"))

own = builders.start("contemporary")    # the shortest path between the contemporary graph's ports
own = builders.move_chain(own)          # carry the pair along it
own = builders.bus_xx(own, on="spare")  # an XX check on a spare path; the builders write its reading
display(own.circuit.draw("mpl", idle_wires=False, fold=30))
same = own.circuit == load_circuit("move-chain-spare-xx-contemporary").circuit
display(Markdown(f"Reading the builders wrote: `{own.analysis()}` · same circuit as case 6's file: {'yes' if same else 'NO'}"))
builder_trial = wb.build("my-case-6", "contemporary", builders.move_chain, (builders.bus_xx, {"on": "spare"}),
                         label="move_chain then bus_xx on a spare path")  # the same steps, as build takes them

<a id="section-2-4"></a>

## 2.4 Checking it before it runs

**`show` puts a circuit on one screen before you spend a shot on it: what it is, where it acts, its gates and what compiling costs.** Three debug views sit one flag away, each folded until you open it: `step=True` steps through the written circuit, `compiled=True` draws what the processor runs, and `timeline=True` shows where each qubit waits.

In [ ]:
wb.show(trial, step=True, compiled=True, timeline=True)

<a id="section-2-5"></a>

## 2.5 Running it

**`run` runs the circuit now, ideal and noisy, and its result shows itself.** Nothing more is written until you keep it ([3.6](workbench.ipynb#section-3-6)). The ideal run must give F = 1 with every shot kept; a warning above the headline says when it doesn't.

In [ ]:
my_run = wb.run(trial)
my_run

**A committed run reads the same way** with `fetch`; here case 3, the benchmark's protected circuit for ibm2016:

In [ ]:
wb.fetch("move-chain", "path-xx-ibm2016")

<a id="section-2-6"></a>

## 2.6 Comparing it with the committed runs

**`compare` sets a run beside committed ones: a table, and a bar per run of its F, with the first run as the reference band.** Below, the example's run beside the organisers' baseline and case 3, which the builders wrote for the same graph.

In [ ]:
wb.compare(wb.fetch("move-chain", "baseline-ibm2016"), wb.fetch("move-chain", "path-xx-ibm2016"), my_run)

**What the builders would have added.** Case 3 reads qubit 3 as vacant before reusing it: a mid-circuit measurement that catches an excitation the move left behind, which a parity check alone cannot see. This hand-written check trusts qubit 3 to be back in |0⟩. That read costs time and needs the mid-circuit measurement channel, which is why the comparison warns that the runs' noise settings differ. At 500 shots per basis, this run's standard error is also several times the committed runs'.

<a id="section-2-7"></a>

## 2.7 The benchmarks

**A benchmark runs many circuits under one set of settings; `benchmark` shows a committed one: a panel of bars per processor on a shared F axis, then its case table.** With several noise profiles, each profile gets its own colour, pale for every shot and solid for kept shots. Below: the move-chain benchmark, the submission's nine cases, then the biased-noise study, a benchmark without cases. A circuit joins one by the steps in [2.8](#section-2-8).

In [ ]:
wb.benchmark("move-chain")

In [ ]:
wb.benchmark("biased-noise")

<a id="section-2-8"></a>

## 2.8 From trial to result

**A circuit you keep goes through five steps, and only the last three write anything git keeps:**

1. `build` saves it as a trial in `workspace/circuits/`, which git ignores. `keep` writes a run of it under `workspace/runs/`, since a run that uses a file outside the repo can't be rebuilt from the repo.
2. `promote` moves the trial into `config/circuit/`.
3. Commit the circuit file on a branch of its own.
4. Add a line naming it to a benchmark file in `config/benchmark/`, as a case if it fills one, and commit that.
5. `make benchmark NAME=<name>` runs every circuit the file lists. Each must pass the self-test first, F = 1 with every shot kept in its ideal run, and the benchmark writes a new folder under `results/runs/`. Commit that folder. The submission reads the committed move-chain benchmark.

The workbench does steps 1, 2 and 4, one cell each: [3.6, Keep it](workbench.ipynb#section-3-6). Committing follows [CONTRIBUTING.md](../CONTRIBUTING.md).

[Back to the top](#top)